# Instagram / X → Aiven MySQL 적재

Instagram과 X의 정규화 CSV만 `daewon` 데이터베이스에 적재합니다. 다른 플랫폼 데이터는 읽거나 변경하지 않습니다. 실제 CSV 헤더를 테이블 컬럼명으로 사용하며 날짜·시간 문자열도 변환하지 않습니다.

In [ ]:
# 최초 1회만 실행하세요.
# %pip install -q pandas sqlalchemy pymysql cryptography python-dotenv

## 1. Aiven 연결

같은 폴더의 `.env`에서 다음 값을 읽습니다: `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DATABASE`.

In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / '02_csv').is_dir() and (candidate / '03_code').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
CSV_ROOT = PROJECT_ROOT / '02_csv'
ENV_PATH = PROJECT_ROOT / '03_code' / '01_python' / '05_merge' / '.env'
load_dotenv(ENV_PATH)
DB = {
    'host': os.getenv('MYSQL_HOST'),
    'port': int(os.getenv('MYSQL_PORT', '3306')),
    'user': os.getenv('MYSQL_USER', 'integral_ej'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': os.getenv('MYSQL_DATABASE', 'daewon'),
}
if not DB['host'] or not DB['password']:
    raise RuntimeError(f'{ENV_PATH}에 MYSQL_HOST와 MYSQL_PASSWORD를 설정하세요.')

db_url = URL.create(
    'mysql+pymysql', username=DB['user'], password=DB['password'],
    host=DB['host'], port=DB['port'], database=DB['database'],
    query={'charset': 'utf8mb4'}
)
engine = create_engine(db_url, pool_pre_ping=True, pool_recycle=300, future=True)
with engine.connect() as conn:
    info = conn.execute(text('SELECT DATABASE(), VERSION()')).one()
print('Aiven MySQL 연결 성공:', info)

Aiven MySQL 연결 성공: ('daewon', '8.4.11')


## 2. CSV 선택 및 읽기

In [2]:
def exactly_one(paths, label):
    paths = sorted(paths)
    if len(paths) != 1:
        raise FileNotFoundError(f'{label}: 파일이 1개여야 하지만 {len(paths)}개입니다: {paths}')
    return paths[0]

instagram_dir = CSV_ROOT / '02_instagram'
x_dir = CSV_ROOT / '03_x' / 'normalized'
FILES = {
    'instagram_keyword': exactly_one(instagram_dir.glob('*3nf_keywords.csv'), 'Instagram 키워드'),
    'instagram_media_type': exactly_one(instagram_dir.glob('*3nf_media_types.csv'), 'Instagram 미디어 유형'),
    'instagram_post': exactly_one(instagram_dir.glob('*3nf_posts.csv'), 'Instagram 게시글'),
    'instagram_post_keyword': exactly_one(instagram_dir.glob('*3nf_post_keywords.csv'), 'Instagram 게시글-키워드'),
    'x_post': exactly_one(x_dir.glob('x_posts.csv'), 'X 게시글'),
    'x_source_file': exactly_one(x_dir.glob('x_source_files.csv'), 'X 원본 파일'),
    'x_post_source': exactly_one(x_dir.glob('x_post_sources.csv'), 'X 게시글 출처'),
    'x_keyword': exactly_one(x_dir.glob('x_keywords.csv'), 'X 키워드'),
    'x_post_keyword': exactly_one(x_dir.glob('x_post_keywords.csv'), 'X 게시글-키워드'),
}

def read_csv_text(path):
    # 날짜·시간과 문자열 'null'을 변환하지 않습니다.
    return pd.read_csv(
        path, dtype='string', encoding='utf-8-sig',
        keep_default_na=False, na_filter=False, on_bad_lines='error'
    )

frames = {table: read_csv_text(path) for table, path in FILES.items()}
display(pd.DataFrame([
    {'table': table, 'rows': len(df), 'columns': ', '.join(df.columns), 'file': str(FILES[table].relative_to(PROJECT_ROOT))}
    for table, df in frames.items()
]))

,table,rows,columns,file
0,instagram_keyword,13,"keyword_id, keyword",02_csv\02_instagram\instagram_normalized_3nf_k...
1,instagram_media_type,3,"media_type_id, media_type",02_csv\02_instagram\instagram_normalized_3nf_m...
2,instagram_post,13464,"post_id, description, media_type_id, post_url,...",02_csv\02_instagram\instagram_normalized_3nf_p...
3,instagram_post_keyword,13464,"post_id, keyword_id",02_csv\02_instagram\instagram_normalized_3nf_p...
4,x_post,5000,"post_id, post_url, account_id, account_name, t...",02_csv\03_x\normalized\x_posts.csv
5,x_source_file,5,"source_file, source_year",02_csv\03_x\normalized\x_source_files.csv
6,x_post_source,5000,"source_file, source_row, post_id, input_row_no",02_csv\03_x\normalized\x_post_sources.csv
7,x_keyword,13,keyword,02_csv\03_x\normalized\x_keywords.csv
8,x_post_keyword,7466,"post_id, keyword_order, keyword",02_csv\03_x\normalized\x_post_keywords.csv


## 3. 테이블 생성

In [3]:
DDL = [
'''CREATE TABLE IF NOT EXISTS instagram_keyword (
 keyword_id INT PRIMARY KEY, keyword VARCHAR(100) NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_media_type (
 media_type_id INT PRIMARY KEY, media_type VARCHAR(30) NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_post (
 post_id BIGINT PRIMARY KEY, description LONGTEXT NULL, media_type_id INT NOT NULL,
 post_url VARCHAR(500) NOT NULL, created_at VARCHAR(40) NOT NULL,
 like_count BIGINT NULL, comment_count BIGINT NOT NULL,
 INDEX idx_instagram_post_media_type (media_type_id),
 CONSTRAINT fk_instagram_post_media_type FOREIGN KEY (media_type_id) REFERENCES instagram_media_type(media_type_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS instagram_post_keyword (
 post_id BIGINT NOT NULL, keyword_id INT NOT NULL, PRIMARY KEY (post_id, keyword_id),
 INDEX idx_ig_pk_keyword (keyword_id),
 CONSTRAINT fk_ig_pk_post FOREIGN KEY (post_id) REFERENCES instagram_post(post_id),
 CONSTRAINT fk_ig_pk_keyword FOREIGN KEY (keyword_id) REFERENCES instagram_keyword(keyword_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post (
 post_id VARCHAR(32) PRIMARY KEY, post_url VARCHAR(500) NOT NULL,
 account_id VARCHAR(255) NOT NULL, account_name VARCHAR(255) NOT NULL,
 text LONGTEXT NOT NULL, created_at VARCHAR(40) NOT NULL,
 comment_count BIGINT NOT NULL, repost_count BIGINT NOT NULL,
 view_count BIGINT NULL, like_count BIGINT NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_source_file (
 source_file VARCHAR(255) PRIMARY KEY, source_year INT NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post_source (
 source_file VARCHAR(255) NOT NULL, source_row BIGINT NOT NULL,
 post_id VARCHAR(32) NOT NULL, input_row_no BIGINT NOT NULL,
 PRIMARY KEY (source_file, source_row), INDEX idx_x_ps_post (post_id),
 CONSTRAINT fk_x_ps_file FOREIGN KEY (source_file) REFERENCES x_source_file(source_file),
 CONSTRAINT fk_x_ps_post FOREIGN KEY (post_id) REFERENCES x_post(post_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_keyword (
 keyword VARCHAR(255) PRIMARY KEY
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4''',
'''CREATE TABLE IF NOT EXISTS x_post_keyword (
 post_id VARCHAR(32) NOT NULL, keyword_order BIGINT NOT NULL, keyword VARCHAR(255) NOT NULL,
 PRIMARY KEY (post_id, keyword_order), INDEX idx_x_pk_keyword (keyword),
 CONSTRAINT fk_x_pk_post FOREIGN KEY (post_id) REFERENCES x_post(post_id),
 CONSTRAINT fk_x_pk_keyword FOREIGN KEY (keyword) REFERENCES x_keyword(keyword)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'''
]
with engine.begin() as conn:
    for statement in DDL:
        conn.execute(text(statement))
    # 이전 실행에서 NOT NULL로 생성된 테이블도 CSV 결측을 보존할 수 있게 변경합니다.
    conn.execute(text('ALTER TABLE instagram_post MODIFY COLUMN like_count BIGINT NULL'))
print('Instagram/X 테이블 생성 완료')

Instagram/X 테이블 생성 완료


## 4. 컬럼 및 PK/FK 검사

In [4]:
EXPECTED_COLUMNS = {
    'instagram_keyword': ['keyword_id', 'keyword'],
    'instagram_media_type': ['media_type_id', 'media_type'],
    'instagram_post': ['post_id', 'description', 'media_type_id', 'post_url', 'created_at', 'like_count', 'comment_count'],
    'instagram_post_keyword': ['post_id', 'keyword_id'],
    'x_post': ['post_id', 'post_url', 'account_id', 'account_name', 'text', 'created_at', 'comment_count', 'repost_count', 'view_count', 'like_count'],
    'x_source_file': ['source_file', 'source_year'],
    'x_post_source': ['source_file', 'source_row', 'post_id', 'input_row_no'],
    'x_keyword': ['keyword'],
    'x_post_keyword': ['post_id', 'keyword_order', 'keyword'],
}
PKS = {
    'instagram_keyword': ['keyword_id'],
    'instagram_media_type': ['media_type_id'],
    'instagram_post': ['post_id'],
    'instagram_post_keyword': ['post_id', 'keyword_id'],
    'x_post': ['post_id'],
    'x_source_file': ['source_file'],
    'x_post_source': ['source_file', 'source_row'],
    'x_keyword': ['keyword'],
    'x_post_keyword': ['post_id', 'keyword_order'],
}

for table, expected in EXPECTED_COLUMNS.items():
    actual = list(frames[table].columns)
    if actual != expected:
        raise ValueError(f'{table} 컬럼 불일치\n기대: {expected}\n실제: {actual}')
    pk_values = frames[table][PKS[table]]
    null_mask = pk_values.apply(lambda column: column.isna() | column.astype('string').str.strip().eq('')).any(axis=1)
    null_rows = frames[table][null_mask]
    duplicate_rows = frames[table][frames[table].duplicated(PKS[table], keep=False)]
    if not null_rows.empty or not duplicate_rows.empty:
        display(null_rows.head(20), duplicate_rows.head(20))
        raise ValueError(
            f'{table} PK 오류: NULL {len(null_rows)}행, 중복 {len(duplicate_rows)}행'
        )

NUMERIC_COLUMNS = {
    'instagram_post': ['post_id', 'media_type_id', 'like_count', 'comment_count'],
    'x_post': ['comment_count', 'repost_count', 'view_count', 'like_count'],
    'x_source_file': ['source_year'],
    'x_post_source': ['source_row', 'input_row_no'],
    'x_post_keyword': ['keyword_order'],
}
NULLABLE_NUMERIC = {('instagram_post', 'like_count'), ('x_post', 'view_count')}
numeric_check = []
for table, columns in NUMERIC_COLUMNS.items():
    for column in columns:
        source = frames[table][column].astype('string')
        blank = source.str.strip().eq('')
        invalid = (~blank) & pd.to_numeric(source.mask(blank), errors='coerce').isna()
        numeric_check.append({
            'table': table, 'column': column,
            'blank_rows': int(blank.sum()), 'invalid_rows': int(invalid.sum())
        })
        if invalid.any():
            display(frames[table].loc[invalid].head(20))
            raise ValueError(f'{table}.{column}에 숫자가 아닌 값이 {invalid.sum()}건 있습니다.')
        if blank.any() and (table, column) not in NULLABLE_NUMERIC:
            display(frames[table].loc[blank].head(20))
            raise ValueError(f'{table}.{column} NOT NULL 숫자 컬럼에 빈 값이 {blank.sum()}건 있습니다.')
display(pd.DataFrame(numeric_check))

def assert_fk(child, child_col, parent, parent_col):
    child_values = set(frames[child][child_col].dropna().astype('string'))
    parent_values = set(frames[parent][parent_col].dropna().astype('string'))
    missing = child_values - parent_values
    if missing:
        raise ValueError(f'{child}.{child_col} FK 불일치 {len(missing)}개: {list(missing)[:5]}')

for relation in [
    ('instagram_post', 'media_type_id', 'instagram_media_type', 'media_type_id'),
    ('instagram_post_keyword', 'post_id', 'instagram_post', 'post_id'),
    ('instagram_post_keyword', 'keyword_id', 'instagram_keyword', 'keyword_id'),
    ('x_post_source', 'source_file', 'x_source_file', 'source_file'),
    ('x_post_source', 'post_id', 'x_post', 'post_id'),
    ('x_post_keyword', 'post_id', 'x_post', 'post_id'),
    ('x_post_keyword', 'keyword', 'x_keyword', 'keyword'),
]:
    assert_fk(*relation)
print('Instagram/X 컬럼 및 PK/FK 검사 통과')

,table,column,blank_rows,invalid_rows
0,instagram_post,post_id,0,0
1,instagram_post,media_type_id,0,0
2,instagram_post,like_count,2164,0
3,instagram_post,comment_count,0,0
4,x_post,comment_count,0,0
5,x_post,repost_count,0,0
6,x_post,view_count,946,0
7,x_post,like_count,0,0
8,x_source_file,source_year,0,0
9,x_post_source,source_row,0,0


Instagram/X 컬럼 및 PK/FK 검사 통과


## 5. Upsert

In [5]:
NULLABLE = {'instagram_post': {'description', 'like_count'}, 'x_post': {'view_count'}}

def db_value(value, nullable):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, str):
        return None if nullable and value == '' else value
    return value.item() if hasattr(value, 'item') else value

def upsert(conn, table, df, pk, chunk_size=300):
    columns = list(df.columns)
    inserts = ', '.join(f'`{column}`' for column in columns)
    params = ', '.join(f':{column}' for column in columns)
    updates = [column for column in columns if column not in pk]
    update_sql = ', '.join(f'`{column}`=VALUES(`{column}`)' for column in updates)
    if not update_sql:
        update_sql = f'`{pk[0]}`=VALUES(`{pk[0]}`)'
    statement = text(
        f'INSERT INTO `{table}` ({inserts}) VALUES ({params}) '
        f'ON DUPLICATE KEY UPDATE {update_sql}'
    )
    nullable = NULLABLE.get(table, set())
    rows = [
        {column: db_value(value, column in nullable) for column, value in row.items()}
        for row in df.to_dict('records')
    ]
    for start in range(0, len(rows), chunk_size):
        conn.execute(statement, rows[start:start + chunk_size])
    return len(rows)

LOAD_ORDER = [
    'instagram_keyword', 'instagram_media_type', 'instagram_post', 'instagram_post_keyword',
    'x_post', 'x_source_file', 'x_keyword', 'x_post_source', 'x_post_keyword',
]
with engine.begin() as conn:
    for table in LOAD_ORDER:
        print(f'{table}: {upsert(conn, table, frames[table], PKS[table]):,}행 처리')
print('Instagram/X 적재 완료')

instagram_keyword: 13행 처리
instagram_media_type: 3행 처리
instagram_post: 13,464행 처리
instagram_post_keyword: 13,464행 처리
x_post: 5,000행 처리
x_source_file: 5행 처리
x_keyword: 13행 처리
x_post_source: 5,000행 처리
x_post_keyword: 7,466행 처리
Instagram/X 적재 완료


## 6. 결과 확인

In [6]:
result = []
with engine.connect() as conn:
    for table in LOAD_ORDER:
        database_rows = conn.execute(text(f'SELECT COUNT(*) FROM `{table}`')).scalar_one()
        source_rows = len(frames[table])
        result.append({
            'table': table, 'source_rows': source_rows,
            'database_rows': database_rows, 'ok': database_rows >= source_rows
        })
result = pd.DataFrame(result)
display(result)
if not result['ok'].all():
    raise RuntimeError('DB 행 수가 원본보다 적은 테이블이 있습니다.')

,table,source_rows,database_rows,ok
0,instagram_keyword,13,13,True
1,instagram_media_type,3,3,True
2,instagram_post,13464,13464,True
3,instagram_post_keyword,13464,13464,True
4,x_post,5000,5000,True
5,x_source_file,5,5,True
6,x_keyword,13,13,True
7,x_post_source,5000,5000,True
8,x_post_keyword,7466,7466,True
